In [3]:
import json
import pandas as pd
import plotly.graph_objects as go

# 1. Buka dan muat file JSON
file_path = '../data/jiat_info.json'
with open(file_path, 'r') as file:
    jiat_info = json.load(file)

# 2. Ekstrak data dan lakukan perhitungan sesuai permintaan
data = []
for pos_name, info in jiat_info.items():
    muka_air_tanah = info.get('Muka Air Tanah', 0)
    kedalaman_sumur = info.get('Kedalaman Sumur', 0)
    
    # Menghitung Data Air Tanah dari selisih (Kedalaman Sumur - Muka Air Tanah)
    data_air_tanah = kedalaman_sumur - muka_air_tanah
    
    data.append({
        'Pos': pos_name,
        'Muka Air Tanah (m)': muka_air_tanah,
        'Data Air Tanah (m)': data_air_tanah,
        'Kedalaman Sumur (m)': kedalaman_sumur
    })

# Konversi ke DataFrame memudahkan visualisasi
df = pd.DataFrame(data)

# 3. Buat Visualisasi menggunakan Plotly
fig = go.Figure()

# Tambahkan nilai Data Air Tanah di bagian bawah barchart (merepresentasikan air)
fig.add_trace(go.Bar(
    x=df['Pos'],
    y=df['Data Air Tanah (m)'],
    name='Data Air Tanah (Tinggi Kolom Air)',
    marker_color='#1f77b4', # Warna biru untuk air
    text=df['Data Air Tanah (m)'].round(2),
    textposition='auto'
))

# Tambahkan nilai Muka Air Tanah bertumpuk di bagian atas (merepresentasikan kekosongan sampai permukaan)
fig.add_trace(go.Bar(
    x=df['Pos'],
    y=df['Muka Air Tanah (m)'],
    name='Muka Air Tanah (Kedalaman Tanah ke Air)',
    marker_color='#d62728', # Warna merah untuk kosong
    text=df['Muka Air Tanah (m)'].round(2),
    textposition='auto'
))

# Kustomisasi Layout Grafik
fig.update_layout(
    title='Visualisasi Muka Air Tanah dan Data Air Tanah Keseluruhan Pos JIAT',
    xaxis_title='Nama Pos (AWLR JIAT)',
    yaxis_title='Kedalaman / Tinggi (Meter)',
    barmode='stack', # Bertumpuk agar total tinggi bar merupakan nilai "Kedalaman Sumur"
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    ),
    template='plotly_white'
)

# Tampilkan plot ke dalam cell notebook
fig.show()

In [12]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import json
import os

# 1. Melakukan load data informasi JIAT
json_path = 'D:/RnD_JIAT/data/jiat_info.json'
with open(json_path, 'r') as f:
    jiat_info = json.load(f)

# Direktori utama tempat csv berada
data_dir = "D:/RnD_JIAT/data"

# 2. Looping melewati setiap pos yang terdaftar di JSON
for pos_name, info in jiat_info.items():
    
    # Ambil Kedalaman Sumur yang spesifik dan unik untuk pos ini
    kedalaman_sumur = info.get('Kedalaman Sumur')
    
    # Konversi string 'Pos AWLR JIAT Curug Agung' ke 'Pos_AWLR_JIAT_Curug_Agung.csv'
    csv_filename = pos_name.replace(" ", "_") + ".csv"
    file_path = os.path.join(data_dir, csv_filename)
    
    # Validasi apakah file csv ada
    if not os.path.exists(file_path):
        print(f"Peringatan: Data sensor untuk {pos_name} tidak ditemukan di lokasi {file_path}. Melewati plot...")
        continue
        
    print(f"\n=======================================================\nMenampilkan Analisis {pos_name} (Kedalaman: {kedalaman_sumur}m)\n=======================================================")

    # Load data dari CSV
    df_eda = pd.read_csv(file_path)

    # Pastikan Waktu ke datetime
    kolom_waktu = "waktu"
    kolom_mat_rerata = "Muka_Air_Tanah_mean"
    df_eda[kolom_waktu] = pd.to_datetime(df_eda[kolom_waktu])

    # Pastikan kolom "Data_Air_Tanah" terkalkulasi jika belum ada di data csv,
    # Menyesuaikan kedalaman_sumur pos ini:
    if "Data_Air_Tanah" not in df_eda.columns:
         df_eda["Data_Air_Tanah"] = kedalaman_sumur - df_eda[kolom_mat_rerata]

    # Membuat Dual Plot (Sumbu Y kiri dan Y kanan)
    fig_dual = make_subplots(specs=[[{"secondary_y": True}]])

    # Trace 1: Muka Air Tanah (TMA) - Sumbu Y Kiri
    fig_dual.add_trace(
        go.Scatter(
            x=df_eda[kolom_waktu],
            y=df_eda[kolom_mat_rerata],
            name="Muka Air Tanah (TMA)",
            mode="lines",
            line=dict(color="#0ea5e9", width=2),
            fill='tozeroy', 
            fillcolor='rgba(14, 165, 233, 0.1)'
        ),
        secondary_y=False
    )

    # Trace 2: Data Air Tanah (Sisa Tinggi Tiang Air) - Sumbu Y Kanan
    fig_dual.add_trace(
        go.Scatter(
            x=df_eda[kolom_waktu],
            y=df_eda["Data_Air_Tanah"],
            name="Data Air Tanah (Vol. Riil)",
            mode="lines",
            line=dict(color="#E69F00", width=2, dash="dot")
        ),
        secondary_y=True
    )

    # ==============================================================================
    # MENGHITUNG DINAMIS RANGE BERDASARKAN MIN/MAX DATA (+ MARGIN 10%)
    # ==============================================================================
    min_tma = df_eda[kolom_mat_rerata].min()
    max_tma = df_eda[kolom_mat_rerata].max()

    # Beri jarak ruang napas (margin offset) 10% di atas dan bawah grafik
    margin_tma = (max_tma - min_tma) * 0.10
    if margin_tma == 0: 
        margin_tma = 1.0 # Antisipasi jika data sensor stagnan sempurna

    # Batas Y-Axis Kiri (TMA) - Dibuat terbalik: Angka besar (dalam) di bawah
    tma_batas_bawah = max_tma + margin_tma 
    tma_batas_atas = min_tma - margin_tma  

    # Batas Y-Axis Kanan (Data Air) - Normal: Angka kecil di bawah
    data_batas_bawah = kedalaman_sumur - tma_batas_bawah
    data_batas_atas = kedalaman_sumur - tma_batas_atas

    # ==============================================================================
    # KOSMETIKA DASHBOARD KHAS DATA SCIENCE
    # ==============================================================================
    fig_dual.update_layout(
        # Judul Dinamis berdasarkan pos loop saat ini
        title=f'<b>Analisis Cermin: {pos_name}</b><br><sup>Fluktuasi Turunnya Ketinggian Air Tanah (TMA vs Riil Volume)</sup>',
        xaxis_title='Waktu Log Kalender',
        template='plotly_dark',
        hovermode='x unified',
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
    )
    
    # Custom Sumbu Kiri (TMA) - Warna Sky Blue (Okabe-Ito)
    fig_dual.update_yaxes(
        title_text="Kedalaman Air Muka (Meter)", 
        range=[tma_batas_bawah, tma_batas_atas], 
        secondary_y=False,
        color="#56B4E9", 
        showgrid=False
    )
    
    # Custom Sumbu Kanan (Data Air Tanah Riil Volumetrik) - Warna Jelas Orange (Okabe-Ito)
    fig_dual.update_yaxes(
        title_text="Sisa Ketinggian Tiang Air (Meter)", 
        range=[data_batas_bawah, data_batas_atas], 
        secondary_y=True,
        color="#E69F00", 
        showgrid=True,
        gridcolor='rgba(255, 255, 255, 0.1)'
    )
    
    # Memunculkan grafik untuk pos iterasi ini
    fig_dual.show()


Menampilkan Analisis Pos AWLR JIAT Curug Agung (Kedalaman: 100.0m)



Menampilkan Analisis Pos AWLR JIAT Sanding (Kedalaman: 80.0m)



Menampilkan Analisis Pos AWLR JIAT Sindang Karya (Kedalaman: 100.0m)



Menampilkan Analisis Pos AWLR JIAT Pondok Kahuru (Kedalaman: 70.0m)



Menampilkan Analisis Pos AWLR JIAT Kamasan (Kedalaman: 100.0m)



Menampilkan Analisis Pos AWLR JIAT Sindang Mandi (Kedalaman: 70.0m)


In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# --- Parameter Simulasi ---
t_discharge = 6.0   # Durasi pemompaan (jam)
t_total = 24.0      # Total durasi simulasi (jam)
dt = 0.1            # Interval waktu (jam)

# Pembuatan array waktu dari 0 s/d 24 jam
waktu = np.arange(0, t_total + dt, dt)

# Parameter hidrologi dummy untuk pendekatan kurva
mat_awal = 12.0     # Muka air tanah awal sebelum dipompa (meter di bawah permukaan tanah)
S_max = 40.0        # Maksimal drawdown/penurunan muka air akibat pemompaan (meter)
k_d = 0.6           # Laju penurunan logaritmik/eksponensial (discharge)
k_r = 0.25          # Laju pemulihan akuifer (recovery)

# --- Perhitungan Muka Air Tanah (MAT) ---
def hitung_penurunan(t):
    if t <= t_discharge:
        # Fase Discharge (Penurunan Muka Air Tanah)
        return S_max * (1 - np.exp(-k_d * t))
    else:
        # Fase Recovery (Pemulihan Muka Air Tanah)
        # s_end adalah titik penurunan terdalam pada saat pompa dimatikan (t = 6)
        s_end = S_max * (1 - np.exp(-k_d * t_discharge))
        return s_end * np.exp(-k_r * (t - t_discharge))

# Aplikasikan fungsi ke seluruh titik waktu
drawdown = np.array([hitung_penurunan(t) for t in waktu])
mat_aktual = mat_awal + drawdown # Kedalaman MAT total

# Membuat DataFrame untuk memudahkan plotting
df = pd.DataFrame({
    'Waktu (Jam)': waktu,
    'MAT (m)': mat_aktual,
})

# --- Visualisasi Menggunakan Plotly ---
fig = go.Figure()

# Filter data untuk masing-masing fase agar warna garis bisa dibedakan
df_discharge = df[df['Waktu (Jam)'] <= t_discharge]
df_recovery = df[df['Waktu (Jam)'] >= t_discharge]

# 1. Tambahkan Trace untuk Fase Discharge
fig.add_trace(go.Scatter(
    x=df_discharge['Waktu (Jam)'],
    y=df_discharge['MAT (m)'],
    mode='lines',
    name='Fase Discharge (0 - 6 Jam)',
    line=dict(color='red', width=3)
))

# 2. Tambahkan Trace untuk Fase Recovery
fig.add_trace(go.Scatter(
    x=df_recovery['Waktu (Jam)'],
    y=df_recovery['MAT (m)'],
    mode='lines',
    name='Fase Recovery (> 6 Jam)',
    line=dict(color='royalblue', width=3)
))

# 3. Tambahkan Garis Penanda Batas Pompa Mati
fig.add_vline(x=t_discharge, line_width=2, line_dash="dash", line_color="green")
fig.add_annotation(
    x=t_discharge + 0.3, 
    y=mat_awal + 2,
    text="Discharge Selesai, Mulai Recovery", 
    showarrow=False, 
    font=dict(color="green", size=12),
    xanchor="left"
)

# 4. Kustomisasi Layout Grafik
fig.update_layout(
    title='Model Siklus Pemompaan (Discharge 6 Jam) & Pemulihan (Recovery) Akuifer',
    xaxis_title='Durasi Waktu T (Jam)',
    yaxis_title='Elevasi Muka Air Tanah Dari Bebas (meter)',
    yaxis=dict(
        autorange="reversed", # Sumbu Y dibalik, karena semakin besar nilai m, semakin dalam di bawah tanah
        zeroline=False
    ),
    template='plotly_white',
    hovermode='x unified',
    legend=dict(
        yanchor="bottom",
        y=0.05,
        xanchor="right",
        x=0.95,
        bgcolor="rgba(255, 255, 255, 0.8)",
        bordercolor="Black",
        borderwidth=1
    )
)

fig.show()

In [8]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# --- 1. Load Data Aktual Curug Agung ---
file_path = r'D:\RnD_JIAT\data\Pos_AWLR_JIAT_Curug_Agung.csv'
df_real = pd.read_csv(file_path)
df_real['waktu'] = pd.to_datetime(df_real['waktu'])

# Beri label fase untuk membedakan asal data
df_real['Kategori_Data'] = 'Aktual Historis'

# --- 2. Buat Data Dummy untuk 14 April 2026 (Resolusi per 1 Jam) ---
waktu_dummy = pd.date_range(start='2026-04-14 00:00:00', end='2026-04-14 23:00:00', freq='h')

# Parameter Model Dummy
t_discharge_jam = 6 
mat_awal = 7.5      # Estimasi awal muka air tanah (disamakan dengan rata-rata Curug Agung ~7.5 meter)
S_max = 1.5         # Drawdown pemompaan 1.5 meter (mat akan turun ke ~9.0 meter)
k_d = 0.5           # Kecepatan drop (discharge)
k_r = 0.3           # Kecepatan naik (recovery)

# Fungsi hitung penurunan (drawdown)
def hitung_penurunan_dummy(t_jam):
    if t_jam <= t_discharge_jam:
        return S_max * (1 - np.exp(-k_d * t_jam))
    else:
        s_end = S_max * (1 - np.exp(-k_d * t_discharge_jam))
        return s_end * np.exp(-k_r * (t_jam - t_discharge_jam))

# Terapkan perhitungan menggunakan index jam 0 s/d 23
jam_indeks = np.arange(len(waktu_dummy))
mat_dummy = np.array([mat_awal + hitung_penurunan_dummy(t) for t in jam_indeks])

# Susun menjadi DataFrame struktur yang sama dengan CSV asli
df_dummy = pd.DataFrame({
    'waktu': waktu_dummy,
    'Muka_Air_Tanah_mean': mat_dummy,
    'Kedalaman_Sumur': 100.0   # Sesuai data curug agung
})

# Hitung sisa kolom berdasarkan rumus
df_dummy['Data_Air_Tanah'] = df_dummy['Kedalaman_Sumur'] - df_dummy['Muka_Air_Tanah_mean']

# Beri label kategori dummy berdasarkan batas pemompaan (6 jam)
df_dummy['Kategori_Data'] = ['Dummy Discharge' if t <= t_discharge_jam else 'Dummy Recovery' for t in jam_indeks]

# --- 3. GABUNGKAN (MERGE) DATA AKTUAL + DUMMY ---
df_gabungan = pd.concat([df_real, df_dummy], ignore_index=True)
df_gabungan = df_gabungan.sort_values(by='waktu').reset_index(drop=True)

# Secara background, data kita kini sudah berisi histori panjang Curug Agung + ekor di tanggal 14 April!

# --- 4. Plot Validasi (Hanya mem-plot tanggal 14 April agar terlihat jelas) ---
df_plot = df_gabungan[df_gabungan['waktu'].dt.date == pd.to_datetime('2026-04-14').date()]
df_disc = df_plot[df_plot['Kategori_Data'] == 'Dummy Discharge']
df_rec  = df_plot[df_plot['Kategori_Data'] == 'Dummy Recovery']

fig = go.Figure()

# Plot Fase Discharge
fig.add_trace(go.Scatter(
    x=df_disc['waktu'], y=df_disc['Muka_Air_Tanah_mean'],
    mode='lines+markers', name='Fase Discharge (Dummy April)', line=dict(color='red', width=3)
))

# Plot Fase Recovery
fig.add_trace(go.Scatter(
    x=df_rec['waktu'], y=df_rec['Muka_Air_Tanah_mean'],
    mode='lines+markers', name='Fase Recovery (Dummy April)', line=dict(color='royalblue', width=3)
))

# Garis Pompa Mati (berada tepat di jam ke-6 dari 00:00, yaitu jam 06:00:00)
waktu_pompa_mati = pd.to_datetime('2026-04-14 06:00:00')
fig.add_vline(x=waktu_pompa_mati, line_width=2, line_dash="dash", line_color="green")
fig.add_annotation(
    x=waktu_pompa_mati, y=mat_awal + 0.2, 
    text="Discharge Selesai, Pompa Mati", showarrow=False, font=dict(color="green")
)

fig.update_layout(
    title='Data Sintetis Tanggal 14 April 2026 yang Di-Append ke CSV Master',
    xaxis_title='Waktu', yaxis_title='Kedalaman Muka Air Tanah (meter)',
    yaxis=dict(autorange="reversed"), template='plotly_white', hovermode='x unified',
    legend=dict(yanchor="bottom", y=0.05, xanchor="right", x=0.95, bgcolor="rgba(255,255,255,0.8)", borderwidth=1)
)

fig.show()

# =========================================================
# JIKA ANDA INGIN MENYIMPAN HASIL GABUNGAN KE CSV BARU:
# df_gabungan.to_csv(r'D:\RnD_JIAT\data\Pos_AWLR_JIAT_Curug_Agung_Update.csv', index=False)
# =========================================================

In [10]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Gunakan dataframe keseluruhan yang memuat ribuan baris data rill + data dummy
df_all = df_gabungan.copy() # atau bisa langsung panggil df_gabungan

# Membuat kerangka grafik dengan dukungan Dual sumbu-Y 
fig = make_subplots(specs=[[{"secondary_y": True}]])

# --- 1. Plot Keseluruhan Muka Air Tanah (Sumbu Y Kiri) ---
fig.add_trace(
    go.Scatter(
        x=df_all['waktu'], 
        y=df_all['Muka_Air_Tanah_mean'],
        mode='lines', # Dihilangkan '+markers' demi kecepatan render data historis
        name='Catatan Historis Muka Air Tanah (MAT)',
        line=dict(color='red', width=1.5),
    ),
    secondary_y=False,
)

# --- 2. Plot Keseluruhan Data Air Tanah / Sisa Kolom (Sumbu Y Kanan) ---
fig.add_trace(
    go.Scatter(
        x=df_all['waktu'], 
        y=df_all['Data_Air_Tanah'],
        mode='lines',
        name='Catatan Historis Cadangan Air Sumur',
        line=dict(color='royalblue', width=1.5),
    ),
    secondary_y=True,
)

# --- Konfigurasi Layout & Sumbu Grafik Visual ---
fig.update_layout(
    title='Tren Lengkap Curug Agung: Hubungan Kedalaman Sumur vs Muka Air (Riil + Dummy)',
    xaxis_title='Timeline Waktu Pengukuran Keseluruhan',
    hovermode='x unified',
    template='plotly_white',
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    )
)

# Update Y Axis (Sumbu Kiri)
fig.update_yaxes(
    title_text="Kedalaman MAT (meter) <br><i>(Terukur dari Sinyal AWLR)</i>", 
    secondary_y=False,
    autorange="reversed", # Dibalik: Grafik ke bawah berarti semakin amblas muka air
    color='red'
)

# Update Y Axis (Sumbu Kanan)
fig.update_yaxes(
    title_text="Kolom Air Aktual (meter) <br><i>(Cadangan)</i>", 
    secondary_y=True,
    color='royalblue'
)

# Anda bisa langsung melakukan zoom-in dan zoom-out secara interaktif!
fig.show()

In [11]:
# Berdasarkan panduan hidrogeologi ("Pendekatan 1 - Recovery rate operasional")
# Ekstraksi Indikator Event Pemompaan pada 14 April 2026 di df_gabungan

import pandas as pd

# 1. Filter khusus Hari Pemompaan (14 April 2026)
df_event = df_gabungan[df_gabungan['waktu'].dt.date == pd.to_datetime('2026-04-14').date()].reset_index(drop=True)

# --- TAHAP C: HITUNG FITUR PER EVENT ---

# Tentukan Parameter Dasar
h_baseline = df_event['Muka_Air_Tanah_mean'].iloc[0] # Baseline awal sebelum pompa nyala (00:00)
h_min = df_event['Muka_Air_Tanah_mean'].max()        # Karena MAT angka kedalaman, titik MAX adalah nilai terdalam
waktu_h_min = df_event.loc[df_event['Muka_Air_Tanah_mean'].idxmax(), 'waktu']

# S_max (Drawdown Maksimum)
s_max = h_min - h_baseline

# Buat DataFrame khusus Fase Recovery (Dari pompa mati hingga akhir observasi)
df_rec_phase = df_event[df_event['waktu'] >= waktu_h_min].copy()

# Hitung R(t) : Persentase Recovery (Berdasarkan rasio h_t terhadap s_max)
# R(t) = ((h_min - h_t) / (h_min - h_eq)) * 100%
df_rec_phase['Recovery_Pcnt'] = ((h_min - df_rec_phase['Muka_Air_Tanah_mean']) / s_max) * 100

# Fungsi untuk mencari waktu tempuh ke N% recovery
def hitung_waktu_t_persen(persentase):
    # Cari baris pertama di saat kesembuhan melewati nilai persentase target
    kondisi = df_rec_phase['Recovery_Pcnt'] >= persentase
    if kondisi.any():
        data_match = df_rec_phase[kondisi].iloc[0]
        # Delta waktu dari saat pompa mati
        waktu_tempuh = data_match['waktu'] - waktu_h_min
        return waktu_tempuh.total_seconds() / 3600 # Ubah ke Jam
    return None

t_50 = hitung_waktu_t_persen(50)
t_80 = hitung_waktu_t_persen(80)
t_90 = hitung_waktu_t_persen(90)

# Residual Drawdown antar siklus (Pergeseran Baseline awal dan akhir)
h_akhir = df_rec_phase['Muka_Air_Tanah_mean'].iloc[-1]
residual_drawdown = h_akhir - h_baseline

# --- CETAK LAPORAN DIAGNOSTIK EVENT ---
print("="*60)
print(" LAPORAN KARAKTERISTIK HYDROGRAPH (EVENT 14-APR-2026)")
print("="*60)
print(f"A. Muka Air Statis (Baseline)  : {h_baseline:.3f} m")
print(f"B. Waktu Recovery Dimulai      : {waktu_h_min.strftime('%H:%M WIB')}")
print(f"C. Drawdown Maksimum (S_max)   : {s_max:.3f} m")
print("-"*60)
print(f"D. Recovery Times (Estimasi Kasar):")
print(f"   - Ke 50% ({h_min - (s_max*0.5):.2f}m)        : {t_50 if t_50 else '> 24'} jam dari pompa mati")
print(f"   - Ke 80% ({h_min - (s_max*0.8):.2f}m)        : {t_80 if t_80 else '> 24'} jam dari pompa mati")
print(f"   - Ke 90% ({h_min - (s_max*0.9):.2f}m)        : {t_90 if t_90 else '> 24'} jam dari pompa mati")
print("-"*60)
print(f"E. Residual Drawdown           : {residual_drawdown:.3f} m")
print("="*60)

# TAHAP E: STATUS IDENTIFIKASI BERDASARKAN KESEHATAN
status = "NORMAL (Pemulihan Cepat)"
if t_90 is None:
    status = "TERTEKAN (Sistem lambat, tak mencapai 90% dalam harian)"
elif residual_drawdown > 0.5:
    status = "WASPADA (Residual drawdown terlalu tinggi, ada efek eksploitasi)"

print(f"Kesimpulan Status Akuifer      : {status}")

 LAPORAN KARAKTERISTIK HYDROGRAPH (EVENT 14-APR-2026)
A. Muka Air Statis (Baseline)  : 7.500 m
B. Waktu Recovery Dimulai      : 06:00 WIB
C. Drawdown Maksimum (S_max)   : 1.425 m
------------------------------------------------------------
D. Recovery Times (Estimasi Kasar):
   - Ke 50% (8.21m)        : 3.0 jam dari pompa mati
   - Ke 80% (7.79m)        : 6.0 jam dari pompa mati
   - Ke 90% (7.64m)        : 8.0 jam dari pompa mati
------------------------------------------------------------
E. Residual Drawdown           : 0.009 m
Kesimpulan Status Akuifer      : NORMAL (Pemulihan Cepat)


## Full EDA Insight

In [ ]:
import pandas as pd
data_filepath = r"D:\RnD_JIAT\data\Pos_AWLR_JIAT_Pondok_Kahuru.csv"
df_c3 = pd.read_csv(data_filepath)
df_c3["waktu"] = pd.to_datetime(df_c3["waktu"])
df_c3 = df_c3.set_index("waktu")
df_c3

# informasi tambahan
# "Kedalaman Sumur": 70.0,
# "Kedalaman Sensor": 60
# Diameter sumur : 8 inch

# ambil data 2026-04-10
# tambahkan data dummy dari sensor pompa dengan nilai 0-1 untuk status penggunaan pompa bisa sesuaikan dari informasi di bawah ini
# cari waktu start pompa jam 4 pagi dan selesai pompa di jam 8 pagi 1st pump
# cari waktu start pompa jam 3 sore dan selesai pompa di jam 6 sore 2nd pump
# output start_pump adalah volume eksploitasi sumur dan durasi discharge

# waktu yang dibutuhkan untuk recovery adalah 7 jam dari jam 8 hinggan jam 3 sore 
# output insight adalah volume m3/jam mAT(m)/jam 
# output minor dalam 1 jam ada kenaikan berapa m  

# tambahan : waktu , muka air tanah , kedalaman sumur : volume tabung

# tambahan dari gw : saat waktu pertama kali discharge dia di mAt / DAT(Data Air Tanah) berapa dan setelah recovery apakah dia kembali ke nilai yang sama jika sama berarti 100 %

# tampilkan mengguakan plotly dengan overlay , muka air tanah, data air tanah, dan status pompa(dummy) 



,Muka_Air_Tanah_mean,Kedalaman_Sumur,Data_Air_Tanah
waktu,,,
2026-01-19 10:00:00,0.78,70.0,69.22
2026-01-19 11:00:00,1.88,70.0,68.12
2026-01-19 12:00:00,1.90,70.0,68.10
2026-01-19 13:00:00,1.90,70.0,68.10
2026-01-19 14:00:00,1.90,70.0,68.10
...,...,...,...
2026-04-13 19:00:00,18.05,70.0,51.95
2026-04-13 20:00:00,18.03,70.0,51.97
2026-04-13 21:00:00,18.03,70.0,51.97


In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import math

# update ke streamlit C3
# untuk sidebar tampilkan kosong
# untuk semua informasi di website atas

# ---------------------------------------------------------
# 1. LOAD DAN FILTER DATA
# ---------------------------------------------------------
data_filepath = r"D:\RnD_JIAT\data\Pos_AWLR_JIAT_Pondok_Kahuru.csv"
df_c3 = pd.read_csv(data_filepath)
df_c3["waktu"] = pd.to_datetime(df_c3["waktu"])
df_c3 = df_c3.set_index("waktu")

# Ambil data spesifik tanggal 2026-04-10
# (Pastikan di dataset Anda sudah mencakup tahun 2026 tanggal 10 April)
df_day = df_c3.loc['2026-04-10'].copy() # 

# Cari nama aktual kolom Muka Air Tanah 
col_mat = 'Muka_Air_Tanah_mean' if 'Muka_Air_Tanah_mean' in df_day.columns else df_day.columns[0]

# ---------------------------------------------------------
# 2. INFORMASI TAMBAHAN & KONSTANTA TANGKI SUMUR
# ---------------------------------------------------------
kedalaman_sumur = 70.0    # meter
kedalaman_sensor = 60.0   # meter
diameter_inch = 8.0       # inch

# Menghitung Luas Penampang Sumur
diameter_m = diameter_inch * 0.0254    # konversi inch ke meter
radius_m = diameter_m / 2
luas_penampang = math.pi * (radius_m ** 2)

# ---------------------------------------------------------
# 3. PENAMBAHAN FITUR: DAT & VOLUME TABUNG
# ---------------------------------------------------------
# Data Air Tanah (DAT) adalah jarak air murni dari dasar sumur
df_day['DAT'] = kedalaman_sumur - df_day[col_mat]
df_day['Volume_Tabung_m3'] = df_day['DAT'] * luas_penampang

# ---------------------------------------------------------
# 4. PENAMBAHAN STATUS POMPA (DUMMY SENSOR)
# ---------------------------------------------------------
df_day['status_pompa'] = 0

# 1st Pump: Start jam 4 pagi -> Selesai jam 8 pagi
pump1_mask = (df_day.index.hour >= 4) & (df_day.index.hour < 8)
df_day.loc[pump1_mask, 'status_pompa'] = 1

# 2nd Pump: Start jam 3 sore -> Selesai jam 6 sore (15:00 - 18:00)
pump2_mask = (df_day.index.hour >= 15) & (df_day.index.hour < 18)
df_day.loc[pump2_mask, 'status_pompa'] = 1

# ---------------------------------------------------------
# 5. EKSTRAKSI INSIGHT (START PUMP & RECOVERY)
# ---------------------------------------------------------
def get_closest_val(target_time, col):
    """Fungsi pembantu untuk mengambil data terdekat di waktu spesifik"""
    target_dt = pd.to_datetime(f'2026-04-10 {target_time}')
    if target_dt in df_day.index:
        res = df_day.loc[target_dt, col]
        return res.iloc[0] if isinstance(res, pd.Series) else res
    idx = df_day.index.get_indexer([target_dt], method='nearest')[0]
    return df_day[col].iloc[idx]

# Ambil poin-poin waktu kritis
mat_start = get_closest_val('04:00:00', col_mat)
dat_start = get_closest_val('04:00:00', 'DAT')
vol_start = get_closest_val('04:00:00', 'Volume_Tabung_m3')

mat_end1 = get_closest_val('08:00:00', col_mat)
dat_end1 = get_closest_val('08:00:00', 'DAT')
vol_end1 = get_closest_val('08:00:00', 'Volume_Tabung_m3')

mat_recov = get_closest_val('15:00:00', col_mat)
dat_recov = get_closest_val('15:00:00', 'DAT')
vol_recov = get_closest_val('15:00:00', 'Volume_Tabung_m3')

# DISCHARGE INSIGHT (1st Pump)
durasi_discharge = 4 # 04:00 - 08:00 (dalam jam)
vol_eksploitasi = vol_start - vol_end1

# RECOVERY INSIGHT
durasi_recovery = 7 # 08:00 - 15:00 (dalam jam)

# Catatan: Muka Air Tanah (mAT) adalah kedalaman dari permukaan. 
# Jika mAT turun angkanya berarti muka air naik, maka perhitungannya dibalik:
kenaikan_mat_total = mat_end1 - mat_recov
kenaikan_vol_total = vol_recov - vol_end1

mat_per_jam = kenaikan_mat_total / durasi_recovery
vol_per_jam = kenaikan_vol_total / durasi_recovery

# Evaluasi 100% Recovery
# Jika level depth di 15:00 lebih kecil/sama dengan (lebih dangkal/sama) dengan initial di 04:00
if mat_recov <= mat_start:
    recovery_status = "100% (Air kembali ke nilai yang sama atau lebih baik / terisi penuh)"
else:
    persentase_recov = (mat_end1 - mat_recov) / (mat_end1 - mat_start) * 100
    recovery_status = f"{persentase_recov:.2f}% (Belum mencapai nilai awal pada saat start pump)"

# Cetak Insight Output
print("="*60)
print("INSIGHT EKSPLOITASI POMPA (1st PUMP)")
print("="*60)
print(f"- Durasi Discharge             : {durasi_discharge} Jam (04:00 s.d 08:00)")
print(f"- Estimasi Volume Eksploitasi  : {vol_eksploitasi:.3f} m³")
print("\n" + "="*60)
print("INSIGHT RECOVERY SUMUR")
print("="*60)
print(f"- Durasi Recovery              : {durasi_recovery} Jam (08:00 s.d 15:00)")
print(f"- Kecepatan Volume Recovery    : {vol_per_jam:.3f} m³/jam")
print(f"- Kecepatan Elevasi Recovery   : {mat_per_jam:.3f} mAT(m)/jam")
print(f"- Output Minor per-Jam         : Terdapat kenaikan muka air tanah sebesar rata-rata {mat_per_jam:.3f} m setiap jam-nya.\n")
print("="*60)
print("RECOVERY vs AWAL POMPA (04:00 vs 15:00)")
print("="*60)
print(f"- Muka Air Tanah Start (04:00) : {mat_start:.2f} m\t(DAT: {dat_start:.2f} m)")
print(f"- Muka Air Tanah Akhir (15:00) : {mat_recov:.2f} m\t(DAT: {dat_recov:.2f} m)")
print(f"- STATUS RECOVERY              : {recovery_status}")
print("="*60)

# ---------------------------------------------------------
# 6. VISUALISASI DENGAN PLOTLY
# ---------------------------------------------------------
import plotly.graph_objects as go

# Hitung min dan max absolut batas Zoom
mat_min = df_day[col_mat].min()
mat_max = df_day[col_mat].max()
pad_mat = (mat_max - mat_min) * 0.1 if (mat_max - mat_min) > 0 else 0.5

dat_min = df_day['DAT'].min()
dat_max = df_day['DAT'].max()
pad_dat = (dat_max - dat_min) * 0.1 if (dat_max - dat_min) > 0 else 0.5

# -------------------------------------------------------------
# PALET WARNA & TEMA PREMIUM (LIGHT MODE)
# -------------------------------------------------------------
color_mat  = "#F97316"   # Oranye dinamis & tegas
color_dat  = "#0284C7"   # Biru profesional (Sky Blue)
color_pump = "#E11D48"   # Merah status mesin (Rose)
bg_color   = "#FFFFFF"   # Putih bersih
grid_color = "#F3F4F6"   # Abu-abu sangat tipis untuk garis grid
text_color = "#374151"   # Abu-abu kehitaman untuk teks
font_family = "Inter, Roboto, sans-serif"

fig = go.Figure()

# 1. Trace mAT di Sumbu Y Utama (Kiri)
fig.add_trace(
    go.Scatter(
        x=df_day.index, 
        y=df_day[col_mat], 
        name=f"mAT ({col_mat})", 
        mode='lines+markers', 
        line=dict(color=color_mat, width=2.5),
        # Desain marker hollow (berongga) menyesuaikan putih background
        marker=dict(size=7, color=bg_color, line=dict(width=2, color=color_mat)),
        yaxis="y1",
        hovertemplate="<b>Waktu:</b> %{x}<br><b>mAT:</b> %{y:.2f} m<extra></extra>"
    )
)

# 2. Trace DAT di Sumbu Y Kedua (Kanan)
fig.add_trace(
    go.Scatter(
        x=df_day.index, 
        y=df_day['DAT'], 
        name="DAT (Kolom Air)", 
        mode='lines', 
        line=dict(color=color_dat, width=2, dash='dashdot'), 
        yaxis="y2",
        hovertemplate="<b>DAT:</b> %{y:.2f} m<extra></extra>"
    )
)

# 3. Trace Status Pompa di Sumbu Y Ketiga (Kanan Luar)
fig.add_trace(
    go.Scatter(
        x=df_day.index, 
        y=df_day['status_pompa'], 
        name="Status Pompa", 
        mode='lines',
        fill='tozeroy', 
        line=dict(color=color_pump, width=1.5), 
        fillcolor='rgba(225, 29, 72, 0.08)', # Warna isi sangat transparan (tipis) agar rapi
        yaxis="y3",
        hoverinfo="skip"
    )
)

# -------------------------------------------------------------
# KONFIGURASI LAYOUT PADA BACKGROUND PUTIH (3 Y-AXIS)
# -------------------------------------------------------------
fig.update_layout(
    title=dict(
        text="<b>Monitoring Hidrologi Eksploitasi & Elevasi Sumur</b>",
        font=dict(size=22, color="#111827", family=font_family),
        x=0.02, 
        y=0.95
    ),
    font=dict(family=font_family, color=text_color),
    paper_bgcolor=bg_color,
    plot_bgcolor=bg_color,
    
    xaxis=dict(
        title="<b>Waktu (Jam)</b>", 
        domain=[0, 0.92], 
        showgrid=True,
        gridcolor=grid_color,
        gridwidth=1.5,
        zeroline=False
    ),
    
    # yaxis (mAT) - KIRI
    yaxis=dict(
        title="<b>Elevasi mAT (meter)</b>", 
        titlefont=dict(color=color_mat, size=13),
        tickfont=dict(color=color_mat),
        showgrid=True,
        gridcolor=grid_color,
        gridwidth=1.5,
        zeroline=False,
        range=[mat_max + pad_mat, mat_min - pad_mat] # REVERSED RANGE
    ),
    
    # yaxis2 (DAT) - KANAN
    yaxis2=dict(
        title="<b>Elevasi DAT (meter)</b>",
        titlefont=dict(color=color_dat, size=13),
        tickfont=dict(color=color_dat),
        overlaying="y",
        side="right",
        showgrid=False, 
        zeroline=False,
        range=[dat_min - pad_dat, dat_max + pad_dat] 
    ),

    # yaxis3 (Pompa) - KANAN LUAR EKSTRA
    yaxis3=dict(
        title="<b>Mesin Pompa</b>",
        titlefont=dict(color=color_pump, size=13),
        tickfont=dict(color=color_pump, size=11),
        overlaying="y",
        side="right",
        position=0.98,
        showgrid=False,
        zeroline=False,
        range=[0, 1.2],
        tickvals=[0, 1],
        ticktext=["OFF", "ON"] 
    ),

    hovermode="x unified",
    # Tooltip bergaya Light Mode
    hoverlabel=dict(
        bgcolor=bg_color,
        font_size=13,
        font_family=font_family,
        font_color=text_color,
        bordercolor="#D1D5DB"
    ),
    
    # Legend menyatu rapi
    legend=dict(
        orientation="h", 
        yanchor="bottom", 
        y=1.05, 
        xanchor="right", 
        x=0.92,
        bgcolor="rgba(255, 255, 255, 0.9)",
        bordercolor=grid_color,
        borderwidth=1,
        font=dict(size=12, color=text_color)
    ),
    margin=dict(l=60, r=60, t=90, b=50) 
)

fig.show()


INSIGHT EKSPLOITASI POMPA (1st PUMP)
- Durasi Discharge             : 4 Jam (04:00 s.d 08:00)
- Estimasi Volume Eksploitasi  : 0.005 m³

INSIGHT RECOVERY SUMUR
- Durasi Recovery              : 7 Jam (08:00 s.d 15:00)
- Kecepatan Volume Recovery    : 0.001 m³/jam
- Kecepatan Elevasi Recovery   : 0.021 mAT(m)/jam
- Output Minor per-Jam         : Terdapat kenaikan muka air tanah sebesar rata-rata 0.021 m setiap jam-nya.

RECOVERY vs AWAL POMPA (04:00 vs 15:00)
- Muka Air Tanah Start (04:00) : 18.01 m	(DAT: 51.99 m)
- Muka Air Tanah Akhir (15:00) : 18.01 m	(DAT: 51.99 m)
- STATUS RECOVERY              : 100% (Air kembali ke nilai yang sama atau lebih baik / terisi penuh)


In [12]:
import pandas as pd
import numpy as np
import math
import plotly.graph_objects as go

# ---------------------------------------------------------
# 1. LOAD DAN FILTER DATA MENGGUNAKAN ST.CACHE
# ---------------------------------------------------------
data_filepath = r"D:\RnD_JIAT\data\Pos_AWLR_JIAT_Pondok_Kahuru.csv"
df_c3 = pd.read_csv(data_filepath)
df_c3["waktu"] = pd.to_datetime(df_c3["waktu"])
df_c3 = df_c3.set_index("waktu")

# Memastikan filter per hari tidak error jika tanggal berbeda
try:
    df_day = df_c3.loc['2026-04-10'].copy()
except KeyError:
    fallback_date = df_c3.index[0].strftime('%Y-%m-%d')
    df_day = df_c3.loc[fallback_date].copy()

# Cari nama aktual kolom Muka Air Tanah 
col_mat = 'Muka_Air_Tanah_mean' if 'Muka_Air_Tanah_mean' in df_day.columns else df_day.columns[0]
target_date_str = str(df_day.index[0].date())

# ---------------------------------------------------------
# 2. INFORMASI TAMBAHAN & KONSTANTA TANGKI SUMUR
# ---------------------------------------------------------
kedalaman_sumur = 70.0    # meter
kedalaman_sensor = 60.0   # meter
diameter_inch = 8.0       # inch

# Menghitung Luas Penampang Sumur
diameter_m = diameter_inch * 0.0254    # konversi inch ke meter
radius_m = diameter_m / 2
luas_penampang = math.pi * (radius_m ** 2)

# ---------------------------------------------------------
# 3. PENAMBAHAN FITUR: DAT & VOLUME TABUNG
# ---------------------------------------------------------
df_day['DAT'] = kedalaman_sumur - df_day[col_mat]
df_day['Volume_Tabung_m3'] = df_day['DAT'] * luas_penampang

# ---------------------------------------------------------
# 4. PENAMBAHAN STATUS POMPA (DUMMY SENSOR)
# ---------------------------------------------------------
df_day['status_pompa'] = 0

pump1_mask = (df_day.index.hour >= 4) & (df_day.index.hour < 8)
df_day.loc[pump1_mask, 'status_pompa'] = 1

pump2_mask = (df_day.index.hour >= 15) & (df_day.index.hour < 18)
df_day.loc[pump2_mask, 'status_pompa'] = 1

# ---------------------------------------------------------
# 5. EKSTRAKSI INSIGHT (START PUMP & RECOVERY)
# ---------------------------------------------------------
def get_closest_val(target_time, col):
    target_dt = pd.to_datetime(f'{target_date_str} {target_time}')
    if target_dt in df_day.index:
        res = df_day.loc[target_dt, col]
        return res.iloc[0] if isinstance(res, pd.Series) else res
    idx = df_day.index.get_indexer([target_dt], method='nearest')[0]
    return df_day[col].iloc[idx]

mat_start = get_closest_val('04:00:00', col_mat)
dat_start = get_closest_val('04:00:00', 'DAT')
vol_start = get_closest_val('04:00:00', 'Volume_Tabung_m3')

mat_end1 = get_closest_val('08:00:00', col_mat)
dat_end1 = get_closest_val('08:00:00', 'DAT')
vol_end1 = get_closest_val('08:00:00', 'Volume_Tabung_m3')

mat_recov = get_closest_val('15:00:00', col_mat)
dat_recov = get_closest_val('15:00:00', 'DAT')
vol_recov = get_closest_val('15:00:00', 'Volume_Tabung_m3')

durasi_discharge = 4 
vol_eksploitasi = vol_start - vol_end1
durasi_recovery = 7 

kenaikan_mat_total = mat_end1 - mat_recov
kenaikan_vol_total = vol_recov - vol_end1
mat_per_jam = kenaikan_mat_total / durasi_recovery
vol_per_jam = kenaikan_vol_total / durasi_recovery

# Evaluasi 100% Recovery
if mat_recov <= mat_start:
    recovery_status = "100% (Terisi Penuh)"
else:
    if (mat_end1 - mat_start) == 0:
        recovery_status = "0.00% (Data identik / fluktuasi error)"
    else:
        persentase_recov = (mat_end1 - mat_recov) / (mat_end1 - mat_start) * 100
        recovery_status = f"{persentase_recov:.2f}% (Belum Kembali Penuh)"

print("="*60)
print("INSIGHT EKSPLOITASI POMPA (1st PUMP)")
print("="*60)
print(f"- Durasi Discharge             : {durasi_discharge} Jam (04:00 s.d 08:00)")
print(f"- Estimasi Volume Eksploitasi  : {vol_eksploitasi:.3f} m³")
print("\n" + "="*60)
print("INSIGHT RECOVERY SUMUR")
print("="*60)
print(f"- Durasi Recovery              : {durasi_recovery} Jam (08:00 s.d 15:00)")
print(f"- Kecepatan Elevasi Recovery   : {mat_per_jam:.3f} mAT(m)/jam")
print(f"- STATUS RECOVERY              : {recovery_status}")

# ---------------------------------------------------------
# 6. VISUALISASI DENGAN PLOTLY
# ---------------------------------------------------------
mat_min = df_day[col_mat].min()
mat_max = df_day[col_mat].max()
pad_mat = (mat_max - mat_min) * 0.1 if (mat_max - mat_min) > 0 else 0.5

dat_min = df_day['DAT'].min()
dat_max = df_day['DAT'].max()
pad_dat = (dat_max - dat_min) * 0.1 if (dat_max - dat_min) > 0 else 0.5

color_mat  = "#F97316"   
color_dat  = "#0284C7"   
color_pump = "#E11D48"   
bg_color   = "#FFFFFF"   
grid_color = "#F3F4F6"   
text_color = "#374151"   
font_family = "Inter, Roboto, sans-serif"

fig = go.Figure()

fig.add_trace(
    go.Scatter(x=df_day.index, y=df_day[col_mat], name=f"mAT ({col_mat})", mode='lines+markers', line=dict(color=color_mat, width=2.5), marker=dict(size=7, color=bg_color, line=dict(width=2, color=color_mat)), yaxis="y1", hovertemplate="<b>Waktu:</b> %{x}<br><b>mAT:</b> %{y:.2f} m<extra></extra>")
)

fig.add_trace(
    go.Scatter(x=df_day.index, y=df_day['DAT'], name="DAT (Kolom Air)", mode='lines', line=dict(color=color_dat, width=2, dash='dashdot'), yaxis="y2", hovertemplate="<b>DAT:</b> %{y:.2f} m<extra></extra>")
)

fig.add_trace(
    go.Scatter(x=df_day.index, y=df_day['status_pompa'], name="Status Pompa", mode='lines', fill='tozeroy', line=dict(color=color_pump, width=1.5), fillcolor='rgba(225, 29, 72, 0.08)', yaxis="y3", hoverinfo="skip")
)

# ---------------------------------------------------------
# 7. INFO FLUKTUASI & PER TAMBAHAN RECOVERY (PLOTLY)
# ---------------------------------------------------------
# Informasi Tambahan Elevasi Tiap Jam saat Recovery (09:00 - 15:00)
for h in range(9, 16):
    t_curr = f"{h:02d}:00:00"
    t_prev = f"{h-1:02d}:00:00"
    
    # Ambil nilai presisi sumbu Y (mAT dan DAT) via method helper
    mat_curr = get_closest_val(t_curr, col_mat)
    mat_prev = get_closest_val(t_prev, col_mat)
    dat_curr = get_closest_val(t_curr, 'DAT')
    dat_prev = get_closest_val(t_prev, 'DAT')
    
    # Hitung nilai mutlak perubahan
    diff_dat = dat_curr - dat_prev
    diff_mat = mat_curr - mat_prev
    
    # Cetak stiker panah di atas garis grafik biru
    sign_dat = "+" if diff_dat > 0 else ""
    sign_mat = "+" if diff_mat > 0 else ""
    text_anno = f"ΔDAT: {sign_dat}{diff_dat:.2f}m<br>ΔmAT: {sign_mat}{diff_mat:.2f}m"
    
    fig.add_annotation(
        x=pd.to_datetime(f"{target_date_str} {t_curr}"),
        y=dat_curr,
        yref="y2",  # Menempel langsung di atas skala DAT (sumbu kanan)
        text=text_anno,
        font=dict(size=9, color="#0284C7"),
        showarrow=True,
        arrowhead=1,
        arrowsize=1.5,
        arrowcolor="#0284C7",
        ax=0,
        ay=-45, # Mendongkrak posisi tag ke atas (supaya tidak numpuk)
        bgcolor="rgba(255, 255, 255, 0.9)",
        bordercolor="#0284C7",
        borderwidth=1,
        borderpad=3
    )

# Shape V-Rect untuk Fenomena Fluktuasi Jam Malam (21:00 - 23:00)
fig.add_vrect(
    x0=pd.to_datetime(f"{target_date_str} 21:00:00"),
    x1=pd.to_datetime(f"{target_date_str} 23:00:00"),
    fillcolor="navy",
    opacity=0.1, # Biru yang teduh
    layer="below", # Di belakang garis utamanya
    line_width=0,
    annotation_text="Fluktuasi Akibat<br>Tekanan Aquifer",
    annotation_position="top left",
    annotation_font_size=11,
    annotation_font_color="navy"
)

# ---------------------------------------------------------
# KONFIGURASI LAYOUT LENGKAP
# ---------------------------------------------------------
fig.update_layout(
    title=dict(text="<b>Monitoring Hidrologi Eksploitasi & Elevasi Sumur</b>", font=dict(size=22, color="#111827", family=font_family), x=0.02, y=0.95),
    font=dict(family=font_family, color=text_color),
    paper_bgcolor=bg_color, plot_bgcolor=bg_color,
    xaxis=dict(title="<b>Waktu (Jam)</b>", domain=[0, 0.92], showgrid=True, gridcolor=grid_color, gridwidth=1.5, zeroline=False),
    yaxis=dict(title="<b>Elevasi mAT (meter)</b>", titlefont=dict(color=color_mat, size=13), tickfont=dict(color=color_mat), showgrid=True, gridcolor=grid_color, gridwidth=1.5, zeroline=False, range=[mat_max + pad_mat, mat_min - pad_mat]),
    yaxis2=dict(title="<b>Elevasi DAT (meter)</b>", titlefont=dict(color=color_dat, size=13), tickfont=dict(color=color_dat), overlaying="y", side="right", showgrid=False, zeroline=False, range=[dat_min - pad_dat, dat_max + pad_dat]),
    yaxis3=dict(title="<b>Mesin Pompa</b>", titlefont=dict(color=color_pump, size=13), tickfont=dict(color=color_pump, size=11), overlaying="y", side="right", position=0.98, showgrid=False, zeroline=False, range=[0, 1.2], tickvals=[0, 1], ticktext=["OFF", "ON"]),
    hovermode="x unified",
    hoverlabel=dict(bgcolor=bg_color, font_size=13, font_family=font_family, font_color=text_color, bordercolor="#D1D5DB"),
    legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="right", x=0.92, bgcolor="rgba(255, 255, 255, 0.9)", bordercolor=grid_color, borderwidth=1, font=dict(size=12, color=text_color)),
    margin=dict(l=60, r=60, t=90, b=50) 
)

fig.show()

INSIGHT EKSPLOITASI POMPA (1st PUMP)
- Durasi Discharge             : 4 Jam (04:00 s.d 08:00)
- Estimasi Volume Eksploitasi  : 0.005 m³

INSIGHT RECOVERY SUMUR
- Durasi Recovery              : 7 Jam (08:00 s.d 15:00)
- Kecepatan Elevasi Recovery   : 0.021 mAT(m)/jam
- STATUS RECOVERY              : 100% (Terisi Penuh)


In [8]:
import plotly.graph_objects as go

# Hitung min dan max absolut batas Zoom
mat_min = df_day[col_mat].min()
mat_max = df_day[col_mat].max()
pad_mat = (mat_max - mat_min) * 0.1 if (mat_max - mat_min) > 0 else 0.5

dat_min = df_day['DAT'].min()
dat_max = df_day['DAT'].max()
pad_dat = (dat_max - dat_min) * 0.1 if (dat_max - dat_min) > 0 else 0.5

# -------------------------------------------------------------
# PALET WARNA & TEMA PREMIUM (LIGHT MODE)
# -------------------------------------------------------------
color_mat  = "#F97316"   # Oranye dinamis & tegas
color_dat  = "#0284C7"   # Biru profesional (Sky Blue)
color_pump = "#E11D48"   # Merah status mesin (Rose)
bg_color   = "#FFFFFF"   # Putih bersih
grid_color = "#F3F4F6"   # Abu-abu sangat tipis untuk garis grid
text_color = "#374151"   # Abu-abu kehitaman untuk teks
font_family = "Inter, Roboto, sans-serif"

fig = go.Figure()

# 1. Trace mAT di Sumbu Y Utama (Kiri)
fig.add_trace(
    go.Scatter(
        x=df_day.index, 
        y=df_day[col_mat], 
        name=f"mAT ({col_mat})", 
        mode='lines+markers', 
        line=dict(color=color_mat, width=2.5),
        # Desain marker hollow (berongga) menyesuaikan putih background
        marker=dict(size=7, color=bg_color, line=dict(width=2, color=color_mat)),
        yaxis="y1",
        hovertemplate="<b>Waktu:</b> %{x}<br><b>mAT:</b> %{y:.2f} m<extra></extra>"
    )
)

# 2. Trace DAT di Sumbu Y Kedua (Kanan)
fig.add_trace(
    go.Scatter(
        x=df_day.index, 
        y=df_day['DAT'], 
        name="DAT (Kolom Air)", 
        mode='lines', 
        line=dict(color=color_dat, width=2, dash='dashdot'), 
        yaxis="y2",
        hovertemplate="<b>DAT:</b> %{y:.2f} m<extra></extra>"
    )
)

# 3. Trace Status Pompa di Sumbu Y Ketiga (Kanan Luar)
fig.add_trace(
    go.Scatter(
        x=df_day.index, 
        y=df_day['status_pompa'], 
        name="Status Pompa", 
        mode='lines',
        fill='tozeroy', 
        line=dict(color=color_pump, width=1.5), 
        fillcolor='rgba(225, 29, 72, 0.08)', # Warna isi sangat transparan (tipis) agar rapi
        yaxis="y3",
        hoverinfo="skip"
    )
)

# -------------------------------------------------------------
# KONFIGURASI LAYOUT PADA BACKGROUND PUTIH (3 Y-AXIS)
# -------------------------------------------------------------
fig.update_layout(
    title=dict(
        text="<b>Monitoring Hidrologi Eksploitasi & Elevasi Sumur</b>",
        font=dict(size=22, color="#111827", family=font_family),
        x=0.02, 
        y=0.95
    ),
    font=dict(family=font_family, color=text_color),
    paper_bgcolor=bg_color,
    plot_bgcolor=bg_color,
    
    xaxis=dict(
        title="<b>Waktu (Jam)</b>", 
        domain=[0, 0.92], 
        showgrid=True,
        gridcolor=grid_color,
        gridwidth=1.5,
        zeroline=False
    ),
    
    # yaxis (mAT) - KIRI
    yaxis=dict(
        title="<b>Elevasi mAT (meter)</b>", 
        titlefont=dict(color=color_mat, size=13),
        tickfont=dict(color=color_mat),
        showgrid=True,
        gridcolor=grid_color,
        gridwidth=1.5,
        zeroline=False,
        range=[mat_max + pad_mat, mat_min - pad_mat] # REVERSED RANGE
    ),
    
    # yaxis2 (DAT) - KANAN
    yaxis2=dict(
        title="<b>Elevasi DAT (meter)</b>",
        titlefont=dict(color=color_dat, size=13),
        tickfont=dict(color=color_dat),
        overlaying="y",
        side="right",
        showgrid=False, 
        zeroline=False,
        range=[dat_min - pad_dat, dat_max + pad_dat] 
    ),

    # yaxis3 (Pompa) - KANAN LUAR EKSTRA
    yaxis3=dict(
        title="<b>Mesin Pompa</b>",
        titlefont=dict(color=color_pump, size=13),
        tickfont=dict(color=color_pump, size=11),
        overlaying="y",
        side="right",
        position=0.98,
        showgrid=False,
        zeroline=False,
        range=[0, 1.2],
        tickvals=[0, 1],
        ticktext=["OFF", "ON"] 
    ),

    hovermode="x unified",
    # Tooltip bergaya Light Mode
    hoverlabel=dict(
        bgcolor=bg_color,
        font_size=13,
        font_family=font_family,
        font_color=text_color,
        bordercolor="#D1D5DB"
    ),
    
    # Legend menyatu rapi
    legend=dict(
        orientation="h", 
        yanchor="bottom", 
        y=1.05, 
        xanchor="right", 
        x=0.92,
        bgcolor="rgba(255, 255, 255, 0.9)",
        bordercolor=grid_color,
        borderwidth=1,
        font=dict(size=12, color=text_color)
    ),
    margin=dict(l=60, r=60, t=90, b=50) 
)

fig.show()

In [6]:
# Hapus penggunaan make_subplots, kita akan buat Multi-Axis kustom lewat go.Figure()
fig = go.Figure()

# Hitung min dan max absolut dari masing-masing dataset untuk batas Zoom
mat_min = df_day[col_mat].min()
mat_max = df_day[col_mat].max()
pad_mat = (mat_max - mat_min) * 0.1 if (mat_max - mat_min) > 0 else 0.5

dat_min = df_day['DAT'].min()
dat_max = df_day['DAT'].max()
pad_dat = (dat_max - dat_min) * 0.1 if (dat_max - dat_min) > 0 else 0.5


# 1. Trace mAT di Sumbu Y Utama (Kiri)
fig.add_trace(
    go.Scatter(
        x=df_day.index, 
        y=df_day[col_mat], 
        name=f"mAT ({col_mat})", 
        mode='lines+markers', 
        line=dict(color='#ff7f0e', width=3),
        marker=dict(size=5),
        yaxis="y1"
    )
)

# 2. Trace DAT di Sumbu Y Kedua (Kanan)
fig.add_trace(
    go.Scatter(
        x=df_day.index, 
        y=df_day['DAT'], 
        name="Data Air Tanah (DAT)", 
        mode='lines+markers', 
        line=dict(color='#1f77b4', width=3, dash='dot'), # Dot dash agar tidak menutup mAT saat tumpang tindih
        marker=dict(symbol='square', size=4),
        yaxis="y2"
    )
)

# 3. Trace Status Pompa di Sumbu Y Ketiga (Kanan Luar Ekstra)
fig.add_trace(
    go.Scatter(
        x=df_day.index, 
        y=df_day['status_pompa'], 
        name="Status Pompa", 
        mode='lines',
        fill='tozeroy', 
        line=dict(color='rgba(255, 0, 0, 0.4)', width=1), 
        fillcolor='rgba(255, 0, 0, 0.15)',
        yaxis="y3",
        hoverinfo="skip"
    )
)

# -------------------------------------------------------------
# KONFIGURASI LAYOUT (3 Y-AXIS)
# -------------------------------------------------------------
fig.update_layout(
    title="Monitoring Hidrologi Eksploitasi & Recovery",
    xaxis=dict(
        title="Waktu", 
        domain=[0, 0.92], # Menyisakan jarak porsi kanan sebesar 8% untuk label axis ke-3
        showgrid=True
    ),
    
    # yaxis (mAT) - Terletak di KIRI
    yaxis=dict(
        title="Elevasi mAT (meter)", 
        titlefont=dict(color="#ff7f0e"),
        tickfont=dict(color="#ff7f0e"),
        showgrid=True,
        # Untuk me-REVERSE sumbu sekaligus membatasi zoom fit
        # kita selipkan nilai min dan max dengan urutan terbalik:
        range=[mat_max + pad_mat, mat_min - pad_mat] 
    ),
    
    # yaxis2 (DAT) - Terletak di KANAN
    yaxis2=dict(
        title="Elevasi DAT (meter)",
        titlefont=dict(color="#1f77b4"),
        tickfont=dict(color="#1f77b4"),
        overlaying="y",
        side="right",
        showgrid=False, # Dimatikan agar garis backgound grid tidak double 
        range=[dat_min - pad_dat, dat_max + pad_dat] # Urutan normal (tidak reversed)
    ),

    # yaxis3 (Pompa) - Terletak di KANAN LUAR (Digeser menjauh)
    yaxis3=dict(
        title="Status Mesin Pompa",
        titlefont=dict(color="red"),
        tickfont=dict(color="red"),
        overlaying="y",
        side="right",
        position=0.98, # Koordinat gesernya
        showgrid=False,
        range=[0, 1.2],
        tickvals=[0, 1]
    ),

    template="plotly_white",
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

fig.show()